# Chorale Workbench: Display One Figured-Bass Chorale

This notebook is the first step of a Colab-friendly workbench for 18th-century chorale harmonization.

It does five things:

1. downloads the Bach Chorale Figured Bass corpus (BCFB v2.0) from Zenodo,
2. extracts and locates the dataset root,
3. loads one chorale with `music21` so we can inspect the score structure computationally,
4. renders the original score with figured bass inline using Verovio's JavaScript toolkit,
5. exports a standalone HTML preview you can open directly in a browser.

Why not use `music21` alone for display?

For this corpus, `music21` reads the notes and parts correctly, but it does not currently surface the MusicXML `<figured-bass>` tags as display-ready figured-bass objects. The notebook therefore uses `music21` for analysis and Verovio for engraving. It now defaults to rendering the original MusicXML directly, with the translated MEI kept as a fallback and comparison path.

In [ ]:
#@title Install dependencies
try:
    import music21  # noqa: F401
except ImportError:
    %pip install -q music21==8.3.0

In [ ]:
#@title Setup helpers (run once)
from __future__ import annotations

import csv
import html
import json
import re
import urllib.request
import zipfile
from pathlib import Path
from typing import Optional
from uuid import uuid4
from xml.etree import ElementTree as ET

from IPython.display import HTML, Javascript, display
from music21 import converter

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

ARCHIVE_URL = "https://zenodo.org/records/5084914/files/juyaolongpaul/Bach_chorale_FB-v2.0.zip?download=1"
ARCHIVE_PATH = DATA_DIR / "bcfb_v2.zip"
EXTRACT_ROOT = DATA_DIR / "bcfb_v2"
DEFAULT_CHORALE = "33.06"

ACCIDENTAL_MAP = {
    "sharp": "#",
    "flat": "b",
    "natural": "nat",
    "double-sharp": "x",
    "double-flat": "bb",
    "cross": "+",
    "backslash": r"\\",
}


In [ ]:
def download_archive(url: str = ARCHIVE_URL, archive_path: Path = ARCHIVE_PATH, force: bool = False) -> Path:
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    if archive_path.exists() and not force:
        print(f"Using cached archive: {archive_path}")
        return archive_path

    print(f"Downloading {url} -> {archive_path}")
    urllib.request.urlretrieve(url, archive_path)
    return archive_path


def extract_archive(archive_path: Path = ARCHIVE_PATH, extract_root: Path = EXTRACT_ROOT, force: bool = False) -> Path:
    if extract_root.exists() and any(extract_root.iterdir()) and not force:
        print(f"Using extracted files in {extract_root}")
        return extract_root

    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_root)
    print(f"Extracted archive into {extract_root}")
    return extract_root


def find_dataset_root(extract_root: Path = EXTRACT_ROOT) -> Path:
    matches = sorted(extract_root.glob("*/reference_table.csv"))
    if not matches:
        raise FileNotFoundError("Could not find reference_table.csv inside the extracted archive.")
    return matches[0].parent


def normalize_example_id(example_id: str) -> str:
    cleaned = example_id.strip().upper().replace("BWV ", "").replace("BWV_", "")
    cleaned = cleaned.removesuffix("_FB")
    return cleaned


def resolve_example_paths(dataset_root: Path, example_id: str = DEFAULT_CHORALE) -> dict[str, Path]:
    normalized = normalize_example_id(example_id)
    musicxml_dir = dataset_root / "FB_source" / "musicXML_master"
    mei_dir = dataset_root / "FB_source" / "translated_mei"

    musicxml_matches = sorted(musicxml_dir.glob(f"BWV_{normalized}*_FB.musicxml"))
    mei_matches = sorted(mei_dir.glob(f"BWV_{normalized}*_FB.musicxml.krn.mei"))

    if not musicxml_matches:
        raise FileNotFoundError(f"No MusicXML file found for example id {example_id!r}.")
    if not mei_matches:
        raise FileNotFoundError(f"No translated MEI file found for example id {example_id!r}.")

    return {
        "musicxml": musicxml_matches[0],
        "mei": mei_matches[0],
    }


def list_example_ids(dataset_root: Path, limit: int = 12) -> list[str]:
    musicxml_dir = dataset_root / "FB_source" / "musicXML_master"
    examples = []
    for path in sorted(musicxml_dir.glob("BWV_*_FB.musicxml")):
        match = re.match(r"BWV_(.+)_FB\.musicxml", path.name)
        if match:
            examples.append(match.group(1))
    return examples[:limit]


def lookup_reference_row(reference_table_path: Path, example_id: str) -> Optional[dict[str, str]]:
    target = normalize_example_id(example_id).replace("*", "")
    with reference_table_path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        for row in reader:
            if not row or not row[0].strip():
                continue
            first_cell = row[0].strip()
            if "Opus BWV" in first_cell or first_cell.endswith("Chorales"):
                continue
            if first_cell.replace("*", "") == target:
                padded = row + [""] * (7 - len(row))
                return {
                    "opus_bwv": padded[0],
                    "breitkopf": padded[1],
                    "cantata": padded[2],
                    "date": padded[3],
                    "text_setting": padded[4],
                    "nba_location": padded[5],
                    "comments": padded[6],
                }
    return None


def _namespace_helper(root: ET.Element):
    if root.tag.startswith("{"):
        namespace = root.tag.split("}", 1)[0] + "}"
    else:
        namespace = ""
    return lambda tag: f"{namespace}{tag}"


def _figure_to_text(figure_element: ET.Element, ns) -> str:
    prefix = ACCIDENTAL_MAP.get(figure_element.findtext(ns("prefix"), default=""), figure_element.findtext(ns("prefix"), default=""))
    number = figure_element.findtext(ns("figure-number"), default="")
    suffix = ACCIDENTAL_MAP.get(figure_element.findtext(ns("suffix"), default=""), figure_element.findtext(ns("suffix"), default=""))
    token = f"{prefix}{number}{suffix}".strip()
    if token:
        return token
    return prefix or suffix or "[alteration]"


def extract_figured_bass_rows(musicxml_path: Path, limit: int = 16) -> list[dict[str, str]]:
    tree = ET.parse(musicxml_path)
    root = tree.getroot()
    ns = _namespace_helper(root)

    score_part_names = {
        score_part.get("id"): score_part.findtext(ns("part-name"), default="")
        for score_part in root.findall(f".//{ns('score-part')}")
    }
    continuo_id = next(
        (part_id for part_id, part_name in score_part_names.items() if part_name.strip().lower() == "continuo"),
        None,
    )

    parts = root.findall(ns("part"))
    if continuo_id is not None:
        parts = [part for part in parts if part.get("id") == continuo_id]

    rows = []
    for part in parts:
        for measure in part.findall(ns("measure")):
            event_index = 0
            for child in list(measure):
                if child.tag != ns("figured-bass"):
                    continue
                event_index += 1
                figure_tokens = [_figure_to_text(figure, ns) for figure in child.findall(ns("figure"))]
                rows.append(
                    {
                        "measure": measure.get("number", "?"),
                        "event_in_measure": str(event_index),
                        "figure": " / ".join(figure_tokens),
                    }
                )
                if len(rows) >= limit:
                    return rows
    return rows


def display_rows_table(rows: list[dict[str, str]]) -> None:
    if not rows:
        display(HTML("<em>No rows found.</em>"))
        return

    columns = list(rows[0].keys())
    header_html = "".join(
        f"<th style='text-align:left; padding:0.35rem 0.5rem; border:1px solid #d0d7de; background:#f6f8fa'>{html.escape(column)}</th>"
        for column in columns
    )
    body_html = "".join(
        "<tr>"
        + "".join(
            f"<td style='padding:0.35rem 0.5rem; border:1px solid #d0d7de'>{html.escape(str(row[column]))}</td>"
            for column in columns
        )
        + "</tr>"
        for row in rows
    )
    display(HTML(f"<table style='border-collapse:collapse'>{'<thead><tr>' + header_html + '</tr></thead><tbody>' + body_html + '</tbody>'}</table>"))


def _verovio_browser_script(mei_string: str, target_id: str, *, scale: int = 34, page_width: int = 1800) -> str:
    return f"""
    (async () => {{
      const target = document.getElementById({json.dumps(target_id)});
      const meiData = {json.dumps(mei_string)};
      const scriptUrl = 'https://www.verovio.org/javascript/latest/verovio-toolkit-wasm.js';

      const loadScriptOnce = (src) => new Promise((resolve, reject) => {{
        const existing = Array.from(document.querySelectorAll('script')).find((element) => element.src === src);
        if (existing) {{
          if (window.verovio && window.verovio.module && window.verovio.module.calledRun) {{
            resolve();
            return;
          }}
          existing.addEventListener('load', () => resolve(), {{ once: true }});
          existing.addEventListener('error', reject, {{ once: true }});
          return;
        }}

        const script = document.createElement('script');
        script.src = src;
        script.async = true;
        script.onload = () => resolve();
        script.onerror = reject;
        document.head.appendChild(script);
      }});

      const waitForToolkit = () => new Promise((resolve, reject) => {{
        const start = Date.now();
        const tick = () => {{
          if (window.verovio && window.verovio.toolkit && window.verovio.module && window.verovio.module.calledRun) {{
            resolve();
            return;
          }}
          if (Date.now() - start > 20000) {{
            reject(new Error('Timed out loading Verovio.'));
            return;
          }}
          setTimeout(tick, 100);
        }};
        tick();
      }});

      try {{
        await loadScriptOnce(scriptUrl);
        await waitForToolkit();

        const toolkit = new verovio.toolkit();
        toolkit.setOptions({{
          scale: {scale},
          pageWidth: {page_width},
          adjustPageHeight: true,
          breaks: 'auto'
        }});
        toolkit.loadData(meiData);

        const pageCount = toolkit.getPageCount();
        let svgPages = '';
        for (let page = 1; page <= pageCount; page += 1) {{
          svgPages += `<div style='margin-bottom: 1.5rem;'>${{toolkit.renderToSVG(page, {{}})}}</div>`;
        }}
        target.innerHTML = svgPages;
      }} catch (error) {{
        target.innerHTML = `<pre>${{String(error)}}</pre>`;
      }}
    }})();
    """


def render_mei_inline(mei_path: Path, *, scale: int = 34, page_width: int = 1800) -> None:
    mei_string = mei_path.read_text(encoding="utf-8")
    target_id = f"verovio-{uuid4().hex}"
    display(HTML(f"<div id='{target_id}'><em>Rendering score...</em></div>"))
    display(Javascript(_verovio_browser_script(mei_string, target_id, scale=scale, page_width=page_width)))


def export_mei_preview_html(mei_path: Path, output_path: Path, *, title: str = "BCFB chorale preview", scale: int = 34, page_width: int = 1800) -> Path:
    mei_string = mei_path.read_text(encoding="utf-8")
    target_id = f"verovio-{uuid4().hex}"
    browser_script = _verovio_browser_script(mei_string, target_id, scale=scale, page_width=page_width)
    html_text = f"""<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>{html.escape(title)}</title>
  <style>
    body {{ font-family: Georgia, 'Times New Roman', serif; margin: 2rem auto; max-width: 1200px; color: #1f2328; }}
    h1 {{ margin-bottom: 0.35rem; }}
    p {{ line-height: 1.5; }}
    .score {{ margin-top: 1.5rem; }}
  </style>
</head>
<body>
  <h1>{html.escape(title)}</h1>
  <p>This file renders the translated MEI score with Verovio from a CDN. Open it in a browser with internet access.</p>
  <div class='score' id='{target_id}'><em>Rendering score...</em></div>
  <script>
{browser_script}
  </script>
</body>
</html>
"""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html_text, encoding="utf-8")
    return output_path


In [ ]:
#@title Rendering helpers for Colab and browser previews
DEFAULT_RENDER_SOURCE = "musicxml"
DEFAULT_VEROVIO_FONT = "Bravura"
DEFAULT_VEROVIO_FONT_FALLBACK = "Bravura"
INPUT_FORMATS = {"musicxml": "xml", "mei": "mei"}


def _verovio_browser_script_for_source(score_data: str, target_id: str, *, input_from: str = "xml", font: str = DEFAULT_VEROVIO_FONT, font_fallback: str = DEFAULT_VEROVIO_FONT_FALLBACK, scale: int = 34, page_width: int = 1800) -> str:
    return f"""
    (async () => {{
      const target = document.getElementById({json.dumps(target_id)});
      const scoreData = {json.dumps(score_data)};
      const scriptUrl = 'https://www.verovio.org/javascript/latest/verovio-toolkit-wasm.js';

      const loadScriptOnce = (src) => new Promise((resolve, reject) => {{
        const existing = Array.from(document.querySelectorAll('script')).find((element) => element.src === src);
        if (existing) {{
          if (window.verovio && window.verovio.module && window.verovio.module.calledRun) {{
            resolve();
            return;
          }}
          existing.addEventListener('load', () => resolve(), {{ once: true }});
          existing.addEventListener('error', reject, {{ once: true }});
          return;
        }}

        const script = document.createElement('script');
        script.src = src;
        script.async = true;
        script.onload = () => resolve();
        script.onerror = reject;
        document.head.appendChild(script);
      }});

      const waitForToolkit = () => new Promise((resolve, reject) => {{
        const start = Date.now();
        const tick = () => {{
          if (window.verovio && window.verovio.toolkit && window.verovio.module && window.verovio.module.calledRun) {{
            resolve();
            return;
          }}
          if (Date.now() - start > 20000) {{
            reject(new Error('Timed out loading Verovio.'));
            return;
          }}
          setTimeout(tick, 100);
        }};
        tick();
      }});

      try {{
        await loadScriptOnce(scriptUrl);
        await waitForToolkit();

        const toolkit = new verovio.toolkit();
        toolkit.setOptions({{
          scale: {scale},
          pageWidth: {page_width},
          adjustPageHeight: true,
          breaks: 'auto',
          inputFrom: {json.dumps(input_from)},
          font: {json.dumps(font)},
          fontFallback: {json.dumps(font_fallback)},
          smuflTextFont: 'embedded'
        }});
        toolkit.loadData(scoreData);

        const pageCount = toolkit.getPageCount();
        let svgPages = '';
        for (let page = 1; page <= pageCount; page += 1) {{
          svgPages += `<div style='margin-bottom: 1.5rem;'>${{toolkit.renderToSVG(page, {{}})}}</div>`;
        }}
        target.innerHTML = svgPages;
      }} catch (error) {{
        target.innerHTML = `<pre>${{String(error)}}</pre>`;
      }}
    }})();
    """


def render_score_inline(score_path: Path, *, input_from: str = "xml", font: str = DEFAULT_VEROVIO_FONT, font_fallback: str = DEFAULT_VEROVIO_FONT_FALLBACK, scale: int = 34, page_width: int = 1800) -> None:
    score_data = score_path.read_text(encoding="utf-8")
    target_id = f"verovio-{uuid4().hex}"
    display(HTML(f"<div id='{target_id}'><em>Rendering score...</em></div>"))
    display(Javascript(_verovio_browser_script_for_source(score_data, target_id, input_from=input_from, font=font, font_fallback=font_fallback, scale=scale, page_width=page_width)))


def export_score_preview_html(score_path: Path, output_path: Path, *, title: str = "BCFB chorale preview", input_from: str = "xml", font: str = DEFAULT_VEROVIO_FONT, font_fallback: str = DEFAULT_VEROVIO_FONT_FALLBACK, scale: int = 34, page_width: int = 1800) -> Path:
    score_data = score_path.read_text(encoding="utf-8")
    target_id = f"verovio-{uuid4().hex}"
    browser_script = _verovio_browser_script_for_source(score_data, target_id, input_from=input_from, font=font, font_fallback=font_fallback, scale=scale, page_width=page_width)
    html_text = f"""<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>{html.escape(title)}</title>
  <style>
    body {{ font-family: Georgia, 'Times New Roman', serif; margin: 2rem auto; max-width: 1200px; color: #1f2328; }}
    h1 {{ margin-bottom: 0.35rem; }}
    p {{ line-height: 1.5; }}
    .score {{ margin-top: 1.5rem; }}
  </style>
</head>
<body>
  <h1>{html.escape(title)}</h1>
  <p>This file renders the score with Verovio from a CDN. Open it in a browser with internet access.</p>
  <div class='score' id='{target_id}'><em>Rendering score...</em></div>
  <script>
{browser_script}
  </script>
</body>
</html>
"""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html_text, encoding="utf-8")
    return output_path


In [ ]:
#@title Download and inspect the corpus
archive_path = download_archive()
extract_root = extract_archive(archive_path)
dataset_root = find_dataset_root(extract_root)

musicxml_count = len(list((dataset_root / "FB_source" / "musicXML_master").glob("BWV_*_FB.musicxml")))
mei_count = len(list((dataset_root / "FB_source" / "translated_mei").glob("BWV_*_FB.musicxml.krn.mei")))

print(f"Dataset root: {dataset_root}")
print(f"MusicXML files: {musicxml_count}")
print(f"Translated MEI files: {mei_count}")
print("First few available examples:", list_example_ids(dataset_root))

In [ ]:
#@title Pick a chorale
example_id = DEFAULT_CHORALE
example_paths = resolve_example_paths(dataset_root, example_id)
reference_row = lookup_reference_row(dataset_root / "reference_table.csv", example_id)

print(f"Example id: BWV {example_id}")
print(f"MusicXML: {example_paths['musicxml']}")
print(f"Translated MEI: {example_paths['mei']}")

if reference_row:
    print("\nReference-table entry:")
    for key, value in reference_row.items():
        if value:
            print(f"- {key}: {value}")

In [ ]:
#@title Inspect score structure with music21
score = converter.parse(example_paths["musicxml"])

part_labels = [part.partName or part.id or f"Part {index + 1}" for index, part in enumerate(score.parts)]
measure_count = len(score.parts[0].getElementsByClass("Measure")) if score.parts else 0
soprano_note_count = len(score.parts[0].recurse().notes) if score.parts else 0
continuo_note_count = len(score.parts[-1].recurse().notes) if score.parts else 0

print(f"music21 parsed {len(score.parts)} parts: {part_labels}")
print(f"Measures in first part: {measure_count}")
print(f"Soprano note count: {soprano_note_count}")
print(f"Continuo note count: {continuo_note_count}")

In [ ]:
#@title Show the first figured-bass events from the MusicXML
figured_bass_rows = extract_figured_bass_rows(example_paths["musicxml"], limit=16)
display_rows_table(figured_bass_rows)

In [ ]:
#@title Choose rendering source
RENDER_SOURCE = DEFAULT_RENDER_SOURCE  #@param ["musicxml", "mei"]
VEROVIO_FONT = DEFAULT_VEROVIO_FONT  #@param ["Bravura", "Leipzig"]
VEROVIO_FONT_FALLBACK = DEFAULT_VEROVIO_FONT_FALLBACK  #@param ["Bravura", "Leipzig"]

render_path = example_paths[RENDER_SOURCE]
render_input_from = INPUT_FORMATS[RENDER_SOURCE]

print(f"Render source: {RENDER_SOURCE}")
print(f"Verovio input mode: {render_input_from}")
print(f"Font: {VEROVIO_FONT}")
print(f"Fallback font: {VEROVIO_FONT_FALLBACK}")
print(f"Render path: {render_path}")

## Render The Original Score

The cell below renders the score in the notebook with Verovio. It defaults to the original MusicXML plus the `Bravura` font, which is a better match for MuseScore-style figured-bass engraving. If you want to compare against the translated MEI path, change `RENDER_SOURCE` above.

In [ ]:
render_score_inline(
    render_path,
    input_from=render_input_from,
    font=VEROVIO_FONT,
    font_fallback=VEROVIO_FONT_FALLBACK,
)

## Export A Browser Preview

This creates a standalone HTML file in `artifacts/`. It is the fastest manual test loop when you want a file you can open directly in a browser outside the notebook.

In [ ]:
preview_path = export_score_preview_html(
    render_path,
    ARTIFACTS_DIR / f"BWV_{normalize_example_id(example_id)}_{RENDER_SOURCE}_preview.html",
    title=f"BCFB Preview: BWV {example_id} ({RENDER_SOURCE})",
    input_from=render_input_from,
    font=VEROVIO_FONT,
    font_fallback=VEROVIO_FONT_FALLBACK,
)
print(f"Wrote preview to: {preview_path.resolve()}")

In [ ]:
#@title Helpers for colored alternate alto and tenor overlays
import copy

from music21 import note, pitch, stream

ALTO_COLOR = "#c2410c"
TENOR_COLOR = "#2563eb"
OVERLAY_PART_IDS = ("P2", "P3")
MUSICXML_DOCTYPE = '<!DOCTYPE score-partwise PUBLIC "-//Recordare//DTD MusicXML 3.1 Partwise//EN" "http://www.musicxml.org/dtds/partwise.dtd">'
ACCIDENTAL_TEXT_FROM_ALTER = {
    -2: "double-flat",
    -1: "flat",
    0: "natural",
    1: "sharp",
    2: "double-sharp",
}
SHARP_ORDER = ["F", "C", "G", "D", "A", "E", "B"]
FLAT_ORDER = ["B", "E", "A", "D", "G", "C", "F"]
REPLACEABLE_MEASURE_TAGS = {"note", "backup", "forward"}


def get_measure_note_rest_events(part) -> list[list[note.GeneralNote]]:
    per_measure = []
    for measure in part.getElementsByClass(stream.Measure):
        source_container = measure.voices[0] if measure.voices else measure
        events = [element for element in source_container if isinstance(element, note.GeneralNote)]
        per_measure.append(events)
    return per_measure


def build_overlay_tokens_from_part(part) -> list[Optional[str]]:
    tokens = []
    for measure_events in get_measure_note_rest_events(part):
        for event in measure_events:
            tokens.append(None if event.isRest else event.nameWithOctave)
    return tokens


def transpose_overlay_tokens(tokens: list[Optional[str]], semitone_edits: dict[int, int]) -> list[Optional[str]]:
    updated = list(tokens)
    for index, semitones in semitone_edits.items():
        token = updated[index]
        if token is None:
            raise ValueError(f"Cannot transpose a rest at token index {index}.")
        updated[index] = pitch.Pitch(token).transpose(semitones).nameWithOctave
    return updated


def make_demo_overlay_tokens(base_score) -> tuple[list[Optional[str]], list[Optional[str]]]:
    alto_tokens = build_overlay_tokens_from_part(base_score.parts[1])
    tenor_tokens = build_overlay_tokens_from_part(base_score.parts[2])

    alto_demo_shifts = {3: 2, 9: -1, 17: 2, 28: -2, 41: 1}
    tenor_demo_shifts = {2: -2, 11: 2, 19: -1, 32: 2, 44: -2}

    return (
        transpose_overlay_tokens(alto_tokens, alto_demo_shifts),
        transpose_overlay_tokens(tenor_tokens, tenor_demo_shifts),
    )


def _make_overlay_event(template_event: note.GeneralNote, token: Optional[str], *, color: str, stem_direction: str):
    if token is None:
        overlay_event = note.Rest()
    else:
        overlay_event = note.Note(token)
        overlay_event.style.color = color
        overlay_event.stemDirection = stem_direction

    overlay_event.duration = copy.deepcopy(template_event.duration)
    if getattr(template_event, "tie", None) is not None and token is not None:
        overlay_event.tie = copy.deepcopy(template_event.tie)
    if getattr(template_event, "beams", None):
        overlay_event.beams = copy.deepcopy(template_event.beams)
    return overlay_event


def _replace_measure_with_two_voices(measure: stream.Measure, measure_tokens: list[Optional[str]], *, color: str, stem_direction: str) -> None:
    source_container = measure.voices[0] if measure.voices else measure
    source_events = [element for element in source_container if isinstance(element, note.GeneralNote)]
    if len(measure_tokens) != len(source_events):
        raise ValueError(
            f"Measure {measure.number} expects {len(source_events)} overlay events but received {len(measure_tokens)}."
        )

    event_offsets = [element.offset for element in source_events]
    voice_one = stream.Voice(id="1")
    voice_two = stream.Voice(id="2")

    for offset, source_event, token in zip(event_offsets, source_events, measure_tokens):
        voice_one.insert(offset, copy.deepcopy(source_event))
        voice_two.insert(
            offset,
            _make_overlay_event(
                source_event,
                token,
                color=color,
                stem_direction=stem_direction,
            ),
        )

    if measure.voices:
        for existing_voice in list(measure.voices):
            measure.remove(existing_voice)
    else:
        for source_event in source_events:
            measure.remove(source_event)

    measure.insert(0, voice_one)
    measure.insert(0, voice_two)


def _add_overlay_voice_to_part(part, tokens: list[Optional[str]], *, color: str, stem_direction: str) -> None:
    measure_events = get_measure_note_rest_events(part)
    flat_event_count = sum(len(events) for events in measure_events)
    if len(tokens) != flat_event_count:
        raise ValueError(f"Overlay token count {len(tokens)} does not match source event count {flat_event_count}.")

    token_index = 0
    for measure, events in zip(part.getElementsByClass(stream.Measure), measure_events):
        measure_tokens = tokens[token_index:token_index + len(events)]
        _replace_measure_with_two_voices(
            measure,
            measure_tokens,
            color=color,
            stem_direction=stem_direction,
        )
        token_index += len(events)


def make_inner_voice_overlay_score(base_score, alto_tokens: list[Optional[str]], tenor_tokens: list[Optional[str]], *, alto_color: str = ALTO_COLOR, tenor_color: str = TENOR_COLOR):
    overlay_score = copy.deepcopy(base_score)
    _add_overlay_voice_to_part(overlay_score.parts[1], alto_tokens, color=alto_color, stem_direction="down")
    _add_overlay_voice_to_part(overlay_score.parts[2], tenor_tokens, color=tenor_color, stem_direction="up")
    return overlay_score


def write_score_to_musicxml(score_to_write, output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    score_to_write.write("musicxml", fp=str(output_path))
    return output_path


def _key_signature_map(fifths: int) -> dict[str, int]:
    mapping = {step: 0 for step in "ABCDEFG"}
    if fifths > 0:
        for step in SHARP_ORDER[:fifths]:
            mapping[step] = 1
    elif fifths < 0:
        for step in FLAT_ORDER[:abs(fifths)]:
            mapping[step] = -1
    return mapping


def _parse_int(text: Optional[str], default: int = 0) -> int:
    if text is None:
        return default
    return int(round(float(text)))


def _has_tie_stop_or_continue(note_element: ET.Element) -> bool:
    tie_types = [tie.get("type") for tie in note_element.findall("tie")]
    notations = note_element.find("notations")
    if notations is not None:
        tie_types.extend(tied.get("type") for tied in notations.findall("tied"))
    return "stop" in tie_types or "continue" in tie_types


def _effective_alter(note_element: ET.Element) -> int:
    return _parse_int(note_element.findtext("pitch/alter"), default=0)


def _collect_measure_note_events(measure: ET.Element) -> list[dict[str, object]]:
    events = []
    position = 0
    xml_index = 0
    for child in list(measure):
        if child.tag == "attributes":
            xml_index += 1
            continue
        if child.tag == "backup":
            position -= _parse_int(child.findtext("duration"))
            xml_index += 1
            continue
        if child.tag == "forward":
            position += _parse_int(child.findtext("duration"))
            xml_index += 1
            continue
        if child.tag != "note":
            xml_index += 1
            continue

        duration = _parse_int(child.findtext("duration"))
        is_chord = child.find("chord") is not None
        onset = events[-1]["onset"] if is_chord and events else position
        events.append(
            {
                "element": child,
                "onset": onset,
                "xml_index": xml_index,
                "is_rest": child.find("rest") is not None,
            }
        )
        if not is_chord:
            position += duration
        xml_index += 1
    return events


def _insert_or_update_accidental(note_element: ET.Element, accidental_text: str) -> None:
    accidental = note_element.find("accidental")
    if accidental is None:
        accidental = ET.Element("accidental")
        insert_index = len(note_element)
        for index, child in enumerate(list(note_element)):
            if child.tag == "type":
                insert_index = index + 1
                break
        note_element.insert(insert_index, accidental)
    accidental.text = accidental_text


def _apply_cross_voice_accidentals(root: ET.Element, *, part_ids: tuple[str, ...] = OVERLAY_PART_IDS) -> None:
    current_fifths_by_part: dict[str, int] = {}
    for part in root.findall("part"):
        part_id = part.get("id", "")
        current_fifths = current_fifths_by_part.get(part_id, 0)
        for measure in part.findall("measure"):
            attributes = measure.find("attributes")
            if attributes is not None and attributes.find("key/fifths") is not None:
                current_fifths = _parse_int(attributes.findtext("key/fifths"), default=current_fifths)
            current_fifths_by_part[part_id] = current_fifths

            if part_id not in part_ids:
                continue

            key_defaults = _key_signature_map(current_fifths)
            measure_state: dict[tuple[str, str], Optional[int]] = {}
            onset_groups: dict[int, list[dict[str, object]]] = {}
            for event in _collect_measure_note_events(measure):
                if event["is_rest"]:
                    continue
                onset_groups.setdefault(int(event["onset"]), []).append(event)

            for onset in sorted(onset_groups):
                group = sorted(onset_groups[onset], key=lambda item: int(item["xml_index"]))
                pitch_groups: dict[tuple[str, str], list[dict[str, object]]] = {}
                for event in group:
                    note_element = event["element"]
                    step = note_element.findtext("pitch/step")
                    octave = note_element.findtext("pitch/octave")
                    if step and octave:
                        pitch_groups.setdefault((step, octave), []).append(event)

                for state_key, pitch_events in pitch_groups.items():
                    step, _ = state_key
                    current_state = measure_state.get(state_key, key_defaults.get(step, 0))
                    actual_alters = {_effective_alter(event["element"]) for event in pitch_events}
                    conflict = len(actual_alters) > 1

                    for event in pitch_events:
                        note_element = event["element"]
                        actual_alter = _effective_alter(note_element)
                        accidental = note_element.find("accidental")
                        needs_accidental = current_state is None or conflict or actual_alter != current_state
                        if _has_tie_stop_or_continue(note_element):
                            needs_accidental = False

                        accidental_text = ACCIDENTAL_TEXT_FROM_ALTER.get(actual_alter)
                        if needs_accidental and accidental_text is not None:
                            _insert_or_update_accidental(note_element, accidental_text)
                        elif accidental is not None:
                            note_element.remove(accidental)

                    measure_state[state_key] = None if conflict else next(iter(actual_alters))


def _overlay_measure_contents(overlay_measure: ET.Element) -> list[ET.Element]:
    return [copy.deepcopy(child) for child in list(overlay_measure) if child.tag in REPLACEABLE_MEASURE_TAGS]


def _replace_measure_music_data(original_measure: ET.Element, overlay_measure: ET.Element) -> None:
    replacement_children = _overlay_measure_contents(overlay_measure)
    original_children = list(original_measure)
    new_children = []
    inserted_replacement = False

    for child in original_children:
        if child.tag in REPLACEABLE_MEASURE_TAGS:
            if not inserted_replacement:
                new_children.extend(copy.deepcopy(element) for element in replacement_children)
                inserted_replacement = True
            continue
        new_children.append(copy.deepcopy(child))

    if not inserted_replacement and replacement_children:
        barline_index = next((index for index, child in enumerate(new_children) if child.tag == "barline"), len(new_children))
        new_children[barline_index:barline_index] = [copy.deepcopy(element) for element in replacement_children]

    for child in list(original_measure):
        original_measure.remove(child)
    for child in new_children:
        original_measure.append(child)


def merge_overlay_parts_into_original_musicxml(original_musicxml_path: Path, overlay_musicxml_path: Path, output_path: Path, *, part_ids: tuple[str, ...] = OVERLAY_PART_IDS) -> Path:
    original_tree = ET.parse(original_musicxml_path)
    overlay_tree = ET.parse(overlay_musicxml_path)

    original_root = original_tree.getroot()
    overlay_root = overlay_tree.getroot()
    original_parts = {part.get("id"): part for part in original_root.findall("part") if part.get("id") in part_ids}
    overlay_parts = {part.get("id"): part for part in overlay_root.findall("part") if part.get("id") in part_ids}

    missing_ids = [part_id for part_id in part_ids if part_id not in overlay_parts or part_id not in original_parts]
    if missing_ids:
        raise ValueError(f"Could not align overlay part ids: {missing_ids}")

    for part_id in part_ids:
        original_measures = original_parts[part_id].findall("measure")
        overlay_measures = overlay_parts[part_id].findall("measure")
        if len(original_measures) != len(overlay_measures):
            raise ValueError(
                f"Part {part_id} has {len(original_measures)} original measures but {len(overlay_measures)} overlay measures."
            )
        for original_measure, overlay_measure in zip(original_measures, overlay_measures):
            _replace_measure_music_data(original_measure, overlay_measure)

    _apply_cross_voice_accidentals(original_root, part_ids=part_ids)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    ET.indent(original_tree, space="  ")
    xml_text = ET.tostring(original_root, encoding="unicode")
    output_path.write_text(
        '<?xml version="1.0" encoding="UTF-8"?>\n'
        + MUSICXML_DOCTYPE
        + "\n"
        + xml_text,
        encoding="utf-8",
    )
    return output_path


## Demo Alternate Alto And Tenor

This section demonstrates the next step in the workbench: take the existing score, add alternate alto and tenor lines as second voices on those staves, color them, and render the combined result. The export path below now keeps the original figured bass intact by merging the corrected alto and tenor parts back into the source MusicXML.


In [ ]:
#@title Build a colored demo overlay score
alto_overlay_tokens, tenor_overlay_tokens = make_demo_overlay_tokens(score)
overlay_score = make_inner_voice_overlay_score(score, alto_overlay_tokens, tenor_overlay_tokens)
voice_layer_musicxml_path = write_score_to_musicxml(
    overlay_score,
    ARTIFACTS_DIR / f"BWV_{normalize_example_id(example_id)}_inner_voice_overlay_voice_layers.musicxml",
)
overlay_musicxml_path = merge_overlay_parts_into_original_musicxml(
    example_paths["musicxml"],
    voice_layer_musicxml_path,
    ARTIFACTS_DIR / f"BWV_{normalize_example_id(example_id)}_inner_voice_overlay.musicxml",
)

print(f"Overlay MusicXML: {overlay_musicxml_path.resolve()}")
print(f"Intermediate voice-layer export: {voice_layer_musicxml_path.resolve()}")
print(f"Alto overlay color: {ALTO_COLOR}")
print(f"Tenor overlay color: {TENOR_COLOR}")


In [ ]:
render_score_inline(
    overlay_musicxml_path,
    input_from="xml",
    font=VEROVIO_FONT,
    font_fallback=VEROVIO_FONT_FALLBACK,
)

In [ ]:
overlay_preview_path = export_score_preview_html(
    overlay_musicxml_path,
    ARTIFACTS_DIR / f"BWV_{normalize_example_id(example_id)}_inner_voice_overlay_preview.html",
    title=f"BCFB Overlay Preview: BWV {example_id}",
    input_from="xml",
    font=VEROVIO_FONT,
    font_fallback=VEROVIO_FONT_FALLBACK,
)
print(f"Wrote overlay preview to: {overlay_preview_path.resolve()}")